Estimate the area of the radio bounds that are within each administrative unit (level=1)

To do: 
* read in country CRS CSV

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import glob

############
# Setup ---
############
# define country
country='nigeria'
itu_table='fem_nigeria'
# Read boundary file
bounds = gpd.read_file('~/Documents/work/AIM Charities/FEM/General Data/HDX Boundaries/nigeria/nga_admbnda_adm1_osgof_20190417.shp')
# assign projection
bounds.crs = 'epsg:4326'
# designate column with admin name
col = 'ADM1_EN'
#designate projection
proj = 'epsg:32632'

# Reproject bounds to a projected CRS for accurate area calculations
bounds_projected = bounds.to_crs(proj)  # UTM Zone 33N for Nigeria

############
# Assignment ---
############
# initialize lists
file_list = []
region_list = []
coverage_list = []

# Get regions
for file in glob.glob(f'output/gpkg/{country}/{itu_table}/*.gpkg'):
    print(file)
    file_list.append(os.path.basename(file))
    gdf = gpd.read_file(file)
    
    # transform CRS to match bounds
    gdf = gdf.to_crs(bounds.crs)
    
    # Project to UTM for accurate area calculations
    gdf_projected = gdf.to_crs(proj)
    
    # spatial join to find intersecting boundaries
    gdf_bounds = gdf.sjoin(bounds, predicate='intersects')
    
    # get unique regions
    unique_regions = gdf_bounds[col].unique()
    region_list.append(unique_regions)
    
    # Calculate coverage percentages for each region
    region_coverage = {}
    for region in unique_regions:
        # Get the boundary polygon for this region
        region_boundary = bounds_projected[bounds_projected[col] == region]
        region_area = region_boundary.geometry.area.sum()
        
        # Get all radio polygons that intersect this region
        radio_polygons = gdf_projected[gdf_projected.index.isin(
            gdf_bounds[gdf_bounds[col] == region].index
        )]
        
        # Calculate intersection area
        intersection = gpd.overlay(radio_polygons, region_boundary, how='intersection')
        intersection_area = intersection.geometry.area.sum()
        
        # Calculate percentage
        coverage_pct = (intersection_area / region_area) * 100
        region_coverage[region] = round(coverage_pct, 2)
    
    coverage_list.append(region_coverage)


    # Create output dataframe
df = pd.DataFrame({
    'file': file_list,
    'regions': region_list
})

df

output/gpkg/nigeria/fem_nigeria/2025-08-12_162648_Crest FM_Ondo.gpkg
output/gpkg/nigeria/fem_nigeria/2025-08-12_161935_precious_Nasagpkga.gpkg
output/gpkg/nigeria/fem_nigeria/2025-08-12_162755_BUZZ FM_Abia.gpkg
output/gpkg/nigeria/fem_nigeria/2025-08-12_184948_Andarza_Jigawa.gpkg
output/gpkg/nigeria/fem_nigeria/2025-08-12_162400_OSBC_Osun.gpkg
output/gpkg/nigeria/fem_nigeria/2025-08-12_162330_Family FM_Ogun.gpkg
output/gpkg/nigeria/fem_nigeria/2025-08-12_162153_faaji_Ogun.gpkg
output/gpkg/nigeria/fem_nigeria/2025-08-12_162435_Freedom_Kano.gpkg


In [ ]:
percents = pd.DataFrame(coverage_list).fillna(0)

df = pd.concat([df, percents], axis=1).drop(columns = ['coverage_percentage'])

In [ ]:
df.to_csv(f'../reach/output/{country}/{country}_station_boundary_names.csv', index=False)